# 1.1 价格与收益率

## 学习目标
- 理解金融资产价格的表示方式
- 掌握简单收益率与对数收益率的计算
- 理解二者的区别与适用场景
- 用 Python 计算真实股票数据的收益率

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'
print('导入完成 ✅')

## 1. 下载真实数据

In [ ]:
# 下载苹果公司 (AAPL) 近两年日线数据
ticker = 'AAPL'
data = yf.download(ticker, start='2022-01-01', end='2024-01-01', progress=False)
price = data['Close'].squeeze()  # 收盘价

print(f'数据形状: {data.shape}')
print(f'时间范围: {price.index[0].date()} → {price.index[-1].date()}')
data[['Open', 'High', 'Low', 'Close', 'Volume']].tail()

In [ ]:
# 绘制收盘价走势
fig, ax = plt.subplots()
ax.plot(price.index, price.values, color='steelblue', linewidth=1.5)
ax.fill_between(price.index, price.values, alpha=0.1, color='steelblue')
ax.set_title(f'{ticker} 收盘价', fontsize=14)
ax.set_ylabel('价格 (USD)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. 简单收益率 (Simple Return)

$$r_t = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1$$

- 直观易懂，代表持有资产的百分比盈亏
- **多资产加总有意义**（可以直接相加做组合收益）
- **跨期相乘**：$n$ 期累积收益 = $(1+r_1)(1+r_2)\cdots(1+r_n) - 1$

In [ ]:
# 计算简单日收益率
simple_ret = price.pct_change().dropna()

print('简单收益率统计：')
print(simple_ret.describe().round(4))
print(f'\n最大单日收益: {simple_ret.max():.2%}')
print(f'最大单日亏损: {simple_ret.min():.2%}')

## 3. 对数收益率 (Log Return)

$$r_t^{\log} = \ln\frac{P_t}{P_{t-1}} = \ln P_t - \ln P_{t-1}$$

- **跨期相加有意义**（直接累加即为累积对数收益）
- 近似正态分布，便于统计建模
- 当 $|r|$ 很小时，$\ln(1+r) \approx r$（两者接近）

In [ ]:
# 计算对数收益率
log_ret = np.log(price / price.shift(1)).dropna()

# 对比两种收益率
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(simple_ret, bins=60, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].set_title('简单收益率分布')
axes[0].set_xlabel('收益率')
axes[0].axvline(simple_ret.mean(), color='red', linestyle='--', label=f'均值={simple_ret.mean():.4f}')
axes[0].legend()

axes[1].hist(log_ret, bins=60, color='darkorange', alpha=0.7, edgecolor='white')
axes[1].set_title('对数收益率分布')
axes[1].set_xlabel('收益率')
axes[1].axvline(log_ret.mean(), color='red', linestyle='--', label=f'均值={log_ret.mean():.4f}')
axes[1].legend()

plt.suptitle(f'{ticker} 日收益率分布对比', fontsize=13)
plt.tight_layout()
plt.show()

## 4. 累积收益率

$$\text{CumReturn}_t = \prod_{i=1}^{t}(1 + r_i) - 1$$

In [ ]:
# 计算累积收益率（假设初始投入 10000 元）
initial_capital = 10000
cumulative_ret = (1 + simple_ret).cumprod()
portfolio_value = cumulative_ret * initial_capital

fig, ax = plt.subplots()
ax.plot(portfolio_value.index, portfolio_value.values, color='green', linewidth=1.5)
ax.axhline(initial_capital, color='gray', linestyle='--', alpha=0.7, label='初始资金')
ax.fill_between(portfolio_value.index, initial_capital, portfolio_value.values,
                where=(portfolio_value.values >= initial_capital),
                alpha=0.2, color='green', label='盈利')
ax.fill_between(portfolio_value.index, initial_capital, portfolio_value.values,
                where=(portfolio_value.values < initial_capital),
                alpha=0.2, color='red', label='亏损')
ax.set_title(f'初始投入 {initial_capital} 元购买 {ticker} 的资产变化', fontsize=13)
ax.set_ylabel('资产价值 (元)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

total_return = cumulative_ret.iloc[-1] - 1
print(f'总收益率: {total_return:.2%}')
print(f'最终资产: {portfolio_value.iloc[-1]:.2f} 元')

## 5. 年化收益率

$$\text{年化收益率} = (1 + \text{总收益率})^{\frac{252}{T}} - 1$$

其中 $T$ 为交易日数，252 为年均交易日。

In [ ]:
T = len(simple_ret)  # 交易日数
total_ret = cumulative_ret.iloc[-1] - 1
annual_ret = (1 + total_ret) ** (252 / T) - 1

print(f'持有交易日数: {T} 天')
print(f'总收益率:     {total_ret:.2%}')
print(f'年化收益率:   {annual_ret:.2%}')

## 6. 🎯 练习

1. 将 `ticker = 'AAPL'` 改为你感兴趣的其他股票（如 `'TSLA'`, `'MSFT'`, `'GOOGL'`），重新运行所有单元格，观察结果差异。
2. 计算 2023 年全年的年化收益率。
3. 对比同一时期多支股票的累积收益率曲线。

---

**下一节** → `02_risk_and_volatility.ipynb`